# A Price or a Ceiling

### Diffusion as transport's far end, and the first requirement no resource can buy

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## The distinction the programme has been circling

Four notebooks have produced the same shape of answer every time. A verb has a requirement; the requirement is
a **floor**; a substrate climbs the floor by spending something that appears in the condition. Cascades pay
$\sec(\pi/N)$, delay pays its exchange rate, resonance pays $2/Q$. Even the one thing that looked like a hard
gate — total dynamical order $\ge3$ — turned out to be a gate only *within* the delay-free rational class, and
dissolved the moment delay was allowed.

So the open question is whether every requirement is like that:

> Is every verb requirement a **price**, which some resource can pay — or are there requirements that are a
> **ceiling**, which no amount of any resource moves?

Two tests, both named by the reviews of the last notebook.

**Test 1 — spatial transport.** A diffusive crossing is neither a lag, nor a delay, nor a resonator. Where does
it fall between $G_{\text{crit}}\to1$ (delay) and $\to0$ (resonance)? The answer turned out to be neither, and
it is a pure number.

**Test 2 — the Copy verb.** Untouched for four notebooks, and the only candidate for a genuine ceiling: no
amount of energy, time, or apparatus lets you clone an unknown quantum state. Is that a very high price, or a
different kind of thing?

---

## Step 1 — Library


In [1]:
import numpy as np
from scipy.optimize import brentq, minimize

# ===========================================================================
# DIFFUSIVE TRANSPORT.  H(s) = exp(-L sqrt((s+gam)/D)) = exp(-l sqrt(s+gam)),  l = L/sqrt(D)
#
# sqrt(gam + i w) = sqrt((r+gam)/2) + i sqrt((r-gam)/2),   r = |gam + i w|
#   phase lag        theta   = l sqrt((r-gam)/2)      -- monotone in w, UNBOUNDED
#   log attenuation  -ln|H|  = l sqrt((r+gam)/2)
#
# Both come from the same square root, so phase and loss are not independent.
# NOTE: np.angle() cannot be used for this phase -- it returns the principal branch,
# and the true diffusive phase runs past pi. See the corrections section.
# ===========================================================================
def diff_phase_atten(l, gam, w):
    r = np.hypot(gam, w)
    return l * np.sqrt((r - gam) / 2), l * np.sqrt((r + gam) / 2)

def diff_Gcrit(l, gam, Phi=np.pi):
    """Phase closure fixes w*; the attenuation there fixes the gain bill."""
    F = lambda w: diff_phase_atten(l, gam, w)[0] - Phi
    hi = 1.0
    while F(hi) < 0:
        hi *= 2
    w = brentq(F, 1e-14, hi, xtol=1e-15, rtol=1e-15)
    return np.exp(diff_phase_atten(l, gam, w)[1]), w


# ===========================================================================
# TELEGRAPHER TRANSPORT.  H(s) = exp(-L sqrt(s(s+2a))/c)
#   a = 0     -> exp(-L s/c): a pure DELAY  (magnitude 1, phase unbounded)
#   a -> inf  -> exp(-L sqrt(2 a s)/c): DIFFUSION with D = c^2/(2a)
# One equation; damping moves between the two.
# ===========================================================================
def tele_phase_atten(L, c, a, w):
    z = -w * w + 2j * a * w                    # s(s+2a) evaluated at s = i w
    k = np.sqrt(np.abs(z)) / c
    th = np.angle(z)                           # in (pi/2, pi] for a >= 0, w > 0
    return L * k * np.sin(th / 2), L * k * np.cos(th / 2)

def tele_Gcrit(L, c, a, Phi=np.pi):
    F = lambda w: tele_phase_atten(L, c, a, w)[0] - Phi
    hi = 1.0
    while F(hi) < 0:
        hi *= 2
    w = brentq(F, 1e-13, hi, xtol=1e-15, rtol=1e-15)
    ph, at = tele_phase_atten(L, c, a, w)
    return np.exp(at), w, at / ph


# ===========================================================================
# Exact finite-domain diffusive transfer, absorbing far end at X:
#   H(s) = sinh((X-L) k) / sinh(X k),   k = sqrt(s/D)
# Used as an independent check on the semi-infinite closed form.
# ===========================================================================
def H_finite(s, L, X, D):
    k = np.sqrt(s / D)
    return np.sinh((X - L) * k) / np.sinh(X * k)

def finite_Gcrit(L, X, D, Phi=np.pi):
    def phase(w):
        val = H_finite(1j * w, L, X, D)
        base = L * np.sqrt(w / (2 * D))        # semi-infinite phase: the correct branch to track
        a = np.angle(val)
        n = np.round((base + a) / (2 * np.pi))
        return -(a - 2 * np.pi * n)
    F = lambda w: phase(w) - Phi
    hi = 1.0
    while F(hi) < 0:
        hi *= 2
    w = brentq(F, 1e-10, hi, xtol=1e-14, rtol=1e-15)
    return 1 / abs(H_finite(1j * w, L, X, D)), w


# ===========================================================================
# Closed-loop reaction-diffusion PDE.  c_t = D c_xx  on [0, X]
#   feedback injection at x=0:  c(0,t) = tanh(-G c(L,t))
#   absorbing far end:          c(X,t) = 0
# ===========================================================================
def pde_swing(G, L=1.0, X=6.0, D=1.0, T=40.0, h=0.02, dtfac=0.2):
    n = int(X / h) + 1; dt = dtfac * h * h / D; steps = int(T / dt)
    x = np.linspace(0, X, n); iL = int(round(L / h))
    c = np.zeros(n); c[1:-1] = 1e-3 * np.sin(np.pi * x[1:-1] / X)
    rec = []; rs = max(1, int(0.002 / dt))
    for k in range(steps):
        c[0] = np.tanh(-G * c[iL]); c[-1] = 0.0
        c[1:-1] += dt * D * (c[2:] - 2 * c[1:-1] + c[:-2]) / h**2
        if k % rs == 0:
            rec.append(c[iL])
    r = np.array(rec[int(0.7 * len(rec)):])
    return float(r.max() - r.min()), np.array(rec), rs * dt


# ===========================================================================
# THE COPY VERB.  Symmetric 1 -> 2 universal qubit cloning.
# The six octahedron states are an exact 2-design for a qubit, so averaging
# fidelity over them equals the Haar average exactly.
# ===========================================================================
SIX = [np.array([1, 0]), np.array([0, 1]),
       np.array([1, 1]) / np.sqrt(2), np.array([1, -1]) / np.sqrt(2),
       np.array([1, 1j]) / np.sqrt(2), np.array([1, -1j]) / np.sqrt(2)]

def unitary_from(p, d):
    n = d * d
    Hm = p[:n].reshape(d, d) + 1j * p[n:2*n].reshape(d, d)
    Hm = Hm + Hm.conj().T
    ev, V = np.linalg.eigh(Hm)
    return V @ np.diag(np.exp(1j * ev)) @ V.conj().T

def clone_fidelity(U, da):
    """BOTH output marginals must approximate |psi>. Scoring only one is not cloning --
       the identity channel would score 1. See the corrections section."""
    F = 0.0
    for psi in SIX:
        v = np.kron(np.kron(psi, np.array([1, 0])), np.eye(da)[0])
        T = (U @ v).reshape(2, 2, da)
        rho1 = np.einsum('ijk,ljk->il', T, T.conj())   # trace out clone 2 and ancilla
        rho2 = np.einsum('jik,jlk->il', T, T.conj())   # trace out clone 1 and ancilla
        F += 0.5 * (np.real(psi.conj() @ rho1 @ psi) + np.real(psi.conj() @ rho2 @ psi))
    return F / len(SIX)

print("library ready")


library ready


---
## Step 2 — Diffusion locks phase to loss

A diffusive crossing has transfer $H(s)=e^{-\ell\sqrt{s}}$, and at $s=i\omega$,
$\sqrt{i\omega}=\sqrt{\omega/2}\,(1+i)$. The real and imaginary parts are **equal**, so

$$\theta = \ell\sqrt{\omega/2} \qquad |H| = e^{-\ell\sqrt{\omega/2}} = e^{-\theta}$$

Phase and attenuation are not two properties of the crossing. They are one number read twice. That is
structurally unlike every crossing so far — a lag has phase bounded and magnitude decaying at a different
rate, a delay has unbounded phase and *no* attenuation, a resonator has magnitude above 1 where its phase is
useful.

Setting $\theta=\pi$ therefore costs exactly $e^{-\pi}$ of magnitude, whatever $\ell$ and $D$ are. With decay
$\gamma$ inside the transport, closing the phase fixes $r=\gamma+2\pi^2/\ell^2$ and the bill becomes

$$\boxed{G_{\text{crit}} = \exp\!\left(\pi\sqrt{1+\mathrm{Da}}\right)}\qquad \mathrm{Da}=\frac{\gamma L^2}{D\pi^2}$$

with floor $e^\pi$ at $\mathrm{Da}\to0$. $\mathrm{Da}$ is the diffusion time over the decay time.


In [2]:
print("Phase and loss are the same variable:  |H| = exp(-theta), exactly.\n")
print(f"{'l':>5} {'w':>7} {'theta':>11} {'|H|':>13} {'exp(-theta)':>13} {'match':>7}")
print("-" * 62)
for l in [0.5, 1.0, 2.0]:
    for w in [0.5, 2.0, 8.0]:
        th, at = diff_phase_atten(l, 0.0, w)
        print(f"{l:>5} {w:>7} {th:>11.6f} {np.exp(-at):>13.9f} {np.exp(-th):>13.9f}"
              f" {str(abs(np.exp(-at)-np.exp(-th))<1e-15):>7}")

print("\n\nClosed form:  G_crit = exp(pi sqrt(1+Da)),   Da = gamma L^2/(D pi^2)\n")
print(f"{'l':>5} {'gamma':>7} {'Da':>10} | {'G_crit solved':>14} {'closed form':>13} {'/e^pi':>9}")
print("-" * 70)
ok = tot = 0
for l in [0.5, 1.0, 3.0]:
    for gam in [0.0, 0.01, 0.1, 1.0, 10.0]:
        Da = gam * l * l / np.pi**2
        G, w = diff_Gcrit(l, max(gam, 1e-14))
        pred = np.exp(np.pi * np.sqrt(1 + Da))
        good = abs(G / pred - 1) < 1e-9; ok += good; tot += 1
        print(f"{l:>5} {gam:>7} {Da:>10.5f} | {G:>14.6f} {pred:>13.6f} {G/np.exp(np.pi):>9.4f}"
              f"  {'ok' if good else 'MISS'}")
print(f"\n   {ok} of {tot} match.   Floor: e^pi = {np.exp(np.pi):.9f}")

print("\n\nIs there a knob? Vary the transport scale over four decades at Da = 0:")
print(f"{'l':>8} {'w* (phase = pi)':>18} {'G_crit':>16}")
print("-" * 46)
for l in [0.01, 0.1, 1.0, 10.0, 100.0]:
    G, w = diff_Gcrit(l, 1e-14)
    print(f"{l:>8} {w:>18.4f} {G:>16.9f}")
print("\n   The operating frequency moves over eight decades. The price does not move at all.")
print("   There is nothing to trade, because phase and loss are the same variable.")


Phase and loss are the same variable:  |H| = exp(-theta), exactly.

    l       w       theta           |H|   exp(-theta)   match
--------------------------------------------------------------
  0.5     0.5    0.250000   0.778800783   0.778800783    True
  0.5     2.0    0.500000   0.606530660   0.606530660    True
  0.5     8.0    1.000000   0.367879441   0.367879441    True
  1.0     0.5    0.500000   0.606530660   0.606530660    True
  1.0     2.0    1.000000   0.367879441   0.367879441    True
  1.0     8.0    2.000000   0.135335283   0.135335283    True
  2.0     0.5    1.000000   0.367879441   0.367879441    True
  2.0     2.0    2.000000   0.135335283   0.135335283    True
  2.0     8.0    4.000000   0.018315639   0.018315639    True


Closed form:  G_crit = exp(pi sqrt(1+Da)),   Da = gamma L^2/(D pi^2)

    l   gamma         Da |  G_crit solved   closed form     /e^pi
----------------------------------------------------------------------
  0.5     0.0    0.00000 |      23.14069

So diffusion is not a cheap phase currency like delay, nor a self-supporting one like resonance. It is the
currency with **a fixed price and no knob**: $e^\pi = 23.1407$, and every radian of phase costs exactly one
neper of loss.

That is worse than a three-stage cascade ($G_{\text{crit}}=8$) and nearly three orders of magnitude worse
than a $Q=50$ resonator ($0.0404$). Diffusion is the most expensive way to close a loop that has been tested.

---
## Step 3 — But it is not a fifth class. It is the far end of one axis.

The telegrapher equation carries a damping parameter $a$ and contains both transport types as limits:

$$H(s)=\exp\!\left(-\frac{L}{c}\sqrt{s(s+2a)}\right) \qquad
\begin{cases} a=0: & e^{-Ls/c}\ \text{— a pure delay}\\[2pt] a\to\infty: & e^{-L\sqrt{2as}/c}\ \text{— diffusion, } D=c^2/2a\end{cases}$$

So the prediction is that $G_{\text{crit}}$ should sweep continuously from $1$ to $e^\pi$ as damping rises, and
the ratio $(\text{attenuation})/(\text{phase})$ — the exchange rate itself — should sweep from $0$ to $1$.


In [3]:
print("One transport equation. Damping moves it from delay to diffusion.\n")
print(f"{'damping a':>12} {'w*':>11} {'atten/phase':>13} {'G_crit':>13}   regime")
print("-" * 68)
for a in [0.0, 0.01, 0.1, 0.5, 2.0, 10.0, 100.0, 1e4, 1e6]:
    G, w, ratio = tele_Gcrit(1.0, 1.0, a)
    reg = ("pure delay" if G < 1.05 else
           "diffusive" if G > 0.999 * np.exp(np.pi) else "in between")
    print(f"{a:>12.4g} {w:>11.4f} {ratio:>13.6f} {G:>13.6f}   {reg}")
print(f"\n   Delay end: G = 1 exactly.   Diffusive end: G = e^pi = {np.exp(np.pi):.6f}.")
print("   The exchange rate atten/phase runs 0 -> 1, and diffusion is where it equals 1.")

print("\n\nWhy the diffusive end cannot be cheaper: no reactive storage, so no resonance.")
print("Diffusion on a bounded domain has strictly real eigenvalues -D k_n^2:")
for M in [4, 8, 16, 32]:
    h = 1.0 / M
    A = (np.diag(-2*np.ones(M)) + np.diag(np.ones(M-1), 1) + np.diag(np.ones(M-1), -1)) / h**2
    A[M-1, M-1] = -1 / h**2                        # no-flux far end
    ev = np.linalg.eigvals(A)
    print(f"   M={M:<4} max |Im(eigenvalue)| = {np.abs(ev.imag).max():.2e}")
print("\n   All real. A first-order-in-time transport equation stores nothing, so |H| < 1 always.")
print("   A wave equation is second order in time and does store -- which is exactly why the")
print("   a = 0 end reaches unity and the diffusive end cannot go below e^pi.")


One transport equation. Damping moves it from delay to diffusion.

   damping a          w*   atten/phase        G_crit   regime
--------------------------------------------------------------------
           0      3.1416      0.000000      1.000000   pure delay
        0.01      3.1416      0.003183      1.010050   pure delay
         0.1      3.1400      0.031815      1.105115   in between
         0.5      3.1025      0.157177      1.638507   in between
           2      2.6501      0.537029      5.403934   in between
          10      0.9416      0.954028     20.028736   in between
         100      0.0986      0.999507     23.104872   in between
       1e+04      0.0010      1.000000     23.140689   diffusive
       1e+06      0.0000      1.000000     23.140693   diffusive

   Delay end: G = 1 exactly.   Diffusive end: G = e^pi = 23.140693.
   The exchange rate atten/phase runs 0 -> 1, and diffusion is where it equals 1.


Why the diffusive end cannot be cheaper: no reactive stor

---
## Step 4 — Two independent checks on $e^\pi$

The closed form came from the semi-infinite transfer. Check it twice: against the exact finite-domain
transfer, and against the actual PDE with the loop closed and a saturating nonlinearity.


In [4]:
L, X, D = 1.0, 6.0, 1.0
G, w = finite_Gcrit(L, X, D)
print("Check 1 -- exact finite-domain transfer sinh((X-L)k)/sinh(Xk), absorbing far end:")
print(f"   L={L}, X={X}, D={D}  (far boundary is 5 diffusion lengths away)")
print(f"   G_crit = {G:.9f}     e^pi = {np.exp(np.pi):.9f}     ratio = {G/np.exp(np.pi):.12f}")
print(f"   w*     = {w:.6f}     2 pi^2/l^2 = {2*np.pi**2:.6f}")

print("\n\nCheck 2 -- the closed-loop PDE, with tanh saturation. Predicted onset at e^pi:")
print(f"{'G':>10} {'G/e^pi':>8} | {'peak-to-peak at x=L':>21} | verdict")
print("-" * 62)
ep = np.exp(np.pi)
for f in [0.7, 0.9, 0.97, 1.03, 1.15, 1.6]:
    Gv = f * ep
    sw, rec, ds = pde_swing(Gv)
    print(f"{Gv:>10.4f} {f:>8.2f} | {sw:>21.3e} | {'OSCILLATES' if sw > 1e-4 else 'decays'}")

sw, rec, ds = pde_swing(1.6 * ep)
r = rec[int(0.7*len(rec)):]; r = r - r.mean()
zc = np.where(np.diff(np.sign(r)) > 0)[0]
per = np.mean(np.diff(zc)) * ds
print(f"\n   Onset bracketed between 0.97 and 1.03 of e^pi.")
print(f"   Measured period above onset {per:.4f} -> omega {2*np.pi/per:.3f}, against w* = {w:.3f}")
print("   (a mature cycle is shifted slightly by its own amplitude, as in the earlier notebooks)")


Check 1 -- exact finite-domain transfer sinh((X-L)k)/sinh(Xk), absorbing far end:
   L=1.0, X=6.0, D=1.0  (far boundary is 5 diffusion lengths away)
   G_crit = 23.140692633     e^pi = 23.140692633     ratio = 1.000000000000
   w*     = 19.739209     2 pi^2/l^2 = 19.739209


Check 2 -- the closed-loop PDE, with tanh saturation. Predicted onset at e^pi:
         G   G/e^pi |   peak-to-peak at x=L | verdict
--------------------------------------------------------------
   16.1985     0.70 |             7.118e-10 | decays
   20.8266     0.90 |             5.376e-10 | decays
   22.4465     0.97 |             4.619e-06 | decays
   23.8349     1.03 |             2.944e-02 | OSCILLATES
   26.6118     1.15 |             5.995e-02 | OSCILLATES
   37.0251     1.60 |             9.299e-02 | OSCILLATES

   Onset bracketed between 0.97 and 1.03 of e^pi.
   Measured period above onset 0.3192 -> omega 19.685, against w* = 19.739
   (a mature cycle is shifted slightly by its own amplitude, as in the e

Ten digits from the finite-domain transfer, and the PDE brackets the onset between $0.97$ and $1.03$ of
$e^\pi$ with the predicted frequency. Transport is settled: **one axis, delay at one end, diffusion at the
other, and the exchange rate is the damping.**

Every requirement so far is still a price.

---
## Step 5 — The Copy verb, and a criterion for telling a price from a ceiling

Here is what all four notebooks have in common, stated so it can be checked: **every threshold has been an
inequality on a magnitude, with a resource inside it.**

$$\left|G\prod_k H_k\right| \ge 1$$

$G$ is in the condition. That is *why* it can be paid — spend more $G$, or build a bond whose $\prod|H_k|$ is
larger. The "gate" on dynamical order looked different, but it was an inequality too ($p\ge3$), and it
dissolved the moment $\tau$ was allowed into the phase sum, because $\tau$ is a resource.

That suggests the criterion:

> A requirement is a **price** if some resource appears in its statement. It is a **ceiling** if none does.

No-cloning is the test case, because its statement contains no resource at all. A unitary cloner would need
$U|\psi\rangle|0\rangle=|\psi\rangle|\psi\rangle$ for all $\psi$; unitarity preserves inner products, so for
any two inputs $x=\langle\psi|\phi\rangle$ must satisfy

$$x = x^2$$

There is no $G$ in that, no $\tau$, no $Q$, no mass, no $\Omega$. Nothing to spend. The only solutions are
$x=0$ and $x=1$ — orthogonal or identical.

But an impossibility proof is not yet a demonstration that resources do not help, since an *approximate*
cloner is allowed. So spend the one resource that could plausibly matter — ancilla room — and watch what the
achievable fidelity does.


In [5]:
print("x = x^2 has no resource in it. Scan the overlap and see:")
print(f"{'overlap x':>11} {'|x - x^2|':>12}")
print("-" * 25)
for x in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    print(f"{x:>11} {abs(x - x*x):>12.4f}")
print("   Zero only at 0 and 1. No parameter anywhere to adjust.\n")

print("Approximate cloning: spend ancilla dimension and see whether the ceiling moves.")
print("Both output marginals are scored, so the identity channel scores 0.75, not 1.\n")
rng = np.random.default_rng(0)
print(f"{'ancilla dim':>12} {'total dim':>10} {'params':>8} {'best fidelity':>15} {'5/6':>10} {'identity':>10}")
print("-" * 70)
for da in [1, 2, 3, 4]:
    d = 2 * 2 * da; best = 0.0
    for _ in range(6):
        p0 = rng.normal(0, 0.6, 2 * d * d)
        r = minimize(lambda p: -clone_fidelity(unitary_from(p, d), da), p0,
                     method='L-BFGS-B', options={'maxiter': 900, 'ftol': 1e-14, 'gtol': 1e-12})
        best = max(best, -r.fun)
    print(f"{da:>12} {d:>10} {2*d*d:>8} {best:>15.8f} {5/6:>10.6f}"
          f" {clone_fidelity(np.eye(d), da):>10.6f}")
print("""
   Ancilla dimension 1 is not enough: 0.819. Dimension 2 reaches 5/6 exactly.
   Dimensions 3 and 4 reach 5/6 -- the same eight digits. The ceiling stops moving.
""")


x = x^2 has no resource in it. Scan the overlap and see:
  overlap x    |x - x^2|
-------------------------
        0.0       0.0000
        0.1       0.0900
        0.3       0.2100
        0.5       0.2500
        0.7       0.2100
        0.9       0.0900
        1.0       0.0000
   Zero only at 0 and 1. No parameter anywhere to adjust.

Approximate cloning: spend ancilla dimension and see whether the ceiling moves.
Both output marginals are scored, so the identity channel scores 0.75, not 1.

 ancilla dim  total dim   params   best fidelity        5/6   identity
----------------------------------------------------------------------
           1          4       32      0.81903559   0.833333   0.750000
           2          8      128      0.83333333   0.833333   0.750000
           3         12      288      0.83333333   0.833333   0.750000
           4         16      512      0.83333333   0.833333   0.750000

   Ancilla dimension 1 is not enough: 0.819. Dimension 2 reaches 5/6 exa

**Both behaviours appear in one table.** Going from one ancilla dimension to two *is* a price: $0.819\to
0.8333$, bought with room. Going from two to four buys nothing — $5/6$ to eight digits, three times over.

That is the shape of a ceiling: a bounded amount of resource reaches it, and past that the resource is simply
not in the condition any more. Contrast $G_{\text{crit}}$, where doubling $Q$ halves the bill forever.

$$\boxed{\;\text{Price: the resource is in the condition, so spending moves the answer without limit.}\;}$$
$$\boxed{\;\text{Ceiling: the resource is absent from the condition, so past a finite point spending does nothing.}\;}$$

---
## Step 6 — Reclassifying the whole programme

With that criterion, the four notebooks sort cleanly, and the sort says something I did not expect.

| Requirement | Statement | Resource inside? | Verdict |
|---|---|---|---|
| Latch onset | $\prod(G_i/\gamma_i)=1$ | $G_i$ | price |
| Clock onset, cascade | $G>\sec(\pi/N)$ | $G$, $N$ | price |
| Clock onset, general | $N\arccos(1/G)+\tau\sqrt{G^2-1}=\Phi$ | $G,N,\tau$ | price |
| Clock onset, resonant | $G>2/Q$ | $Q$ | price |
| Clock onset, diffusive | $G>e^{\pi\sqrt{1+\mathrm{Da}}}$ | $\mathrm{Da}$ only — and $e^\pi$ is a floor $\mathrm{Da}$ cannot lower | **price with a floor** |
| Order $\ge3$ | $p\pi/2>\pi$ | $p$, and $\tau$ once delay is allowed | price, **class-relative** |
| Hold, conserved mass | $M>M_c$ | $M$ | price |
| Noise as a clock | $\mathrm{CV}\sim\Omega^{-1/2}$ | $\Omega$ — but it works *against* you | price you cannot win |
| **Copy an unknown state** | $x=x^2$ | **none** | **ceiling** |

Two entries are worth dwelling on.

**Diffusion is the first floor.** $e^\pi$ is not a ceiling — you can still pay it, and $\mathrm{Da}$ can make
it worse — but no parameter of diffusion makes it *better*. It is a price with a minimum, and the minimum is
structural: phase and loss come out of the same square root. That is a new category between "price" and
"ceiling," and it exists because the two bills are carried by one variable.

**Every gate found before now was class-relative.** Order $\ge3$ felt absolute while the class was
delay-free rational functions; it dissolved when the class grew. That should make anyone suspicious of the
other gates — and the criterion explains why no-cloning is different in kind rather than merely in degree. Its
condition is an equality on an inner product, dimensionless and scale-free. There is no larger class to escape
into, because enlarging the class means adding resources, and resources are exactly what the condition
does not mention.

$$\boxed{\;\text{Learning finds cheaper bonds. It does not find a bond where } x \ne x^2.\;}$$

---
## Step 7 — Corrections

**A phrase from the last notebook needs pairing.** I wrote that a resonant crossing "supplies the gain." That
is operationally right and physically incomplete: a passive resonator creates nothing. It stores energy in its
reactive degrees of freedom and returns it coherently, with loss set by $\zeta$. The external element still
replaces what is lost, indefinitely. The accurate version: **a high-$Q$ bond lowers the external restoring gain
because it stores and coherently returns a distinction across time.** That strengthens the access reading
rather than weakening it — bond quality *is* the ability to return what crossed.

**Two bugs this session, both caught by an impossible number.**

1. `np.angle` returned the principal branch for the diffusive phase, which runs past $\pi$. At $\ell=2,\omega=8$
   the true phase is $4$ rad and the principal branch reported $-2.28$, which made the magnitude–phase identity
   read as violated and sent the root finder to $\omega=4\times10^6$ with $G_{\text{crit}}=\infty$. Fixed by
   using the analytic branch $\ell\sqrt{(r-\gamma)/2}$, which is monotone by construction.
2. The cloning figure of merit scored only output 1, so the **identity channel scored 1.000** — a perfect
   cloner that copies nothing. Fixed by scoring both marginals.

Both errors announced themselves as results that were too good. That is the third distinct failure mode in the
programme's error log, alongside the mis-indexed steady state and the biased sampler — and all three were found
the same way: *a number that could not be right.*

**And the standing method rule holds again.** Every threshold here was bracketed with runs long enough to
separate a slow transient from a cycle; the PDE at $0.97\,e^\pi$ reads $4.6\times10^{-6}$ rather than zero,
which is decay caught mid-flight, not oscillation.

---
## Step 8 — Scoring

| Prediction | Result | Number |
|---|---|---|
| Diffusion locks phase to loss | **confirmed** | $\lvert H\rvert = e^{-\theta}$ to machine precision |
| Diffusive bill has a closed form | **derived and verified** | $\exp(\pi\sqrt{1+\mathrm{Da}})$, 15 of 15 |
| Diffusion has no knob | **confirmed** | $\omega^*$ over 8 decades, price fixed at 23.140693 |
| Transport is one axis, not two classes | **confirmed** | telegrapher sweeps $G_{\text{crit}}$ from 1 to $e^\pi$ |
| Diffusion cannot resonate | **confirmed** | all eigenvalues real, $\lvert H\rvert<1$ always |
| $e^\pi$ survives the exact finite domain | **confirmed** | ratio $1.000000000000$ |
| $e^\pi$ governs the nonlinear PDE | **confirmed** | onset bracketed in $[0.97,1.03]\,e^\pi$ |
| Copy is a ceiling, not a price | **confirmed** | $5/6$ to 8 digits at ancilla dim 2, 3 and 4 |

**What is not new.** None of the mathematics. $e^{-\ell\sqrt{s}}$ is the standard semi-infinite diffusive
transfer and its equal real and imaginary parts are textbook; the telegrapher limits are textbook; $5/6$ is
Bužek–Hillery, proved in 1996. A process-control engineer would recognize "diffusive lag is the hardest thing
to stabilize" as daily experience.

**What the exercise established.** The criterion. Four notebooks of finding prices made it easy to assume
everything is a price, and the way to test that was to look for a requirement whose *statement contains no
resource*. That criterion was not available until enough prices had been catalogued to see what they had in
common. It then sorted the whole programme, found a new middle category (a floor: diffusion), and identified
why one requirement is different in kind rather than expensive.

---
## Step 9 — Still open

- **Is the floor/ceiling distinction stable?** Order $\ge3$ looked absolute until delay was allowed. The
  honest position is that $e^\pi$ is a floor *for parabolic transport*, and the telegrapher result already shows
  that leaving that class escapes it. Whether some enlargement of class escapes $x=x^2$ is the question, and
  the criterion says no, but a criterion is not a proof.
- **Do other verbs have ceilings?** Copy has one. The programme has tested Hold, Clock, Threshold, Gain and
  Sign, and every one was a price. Erase is the untested candidate: Landauer's $kT\ln2$ has $T$ in it, so by
  the criterion it should be a price that goes to zero with temperature — and it does, which is a check the
  criterion passes without being tuned.
- **Global structure**, still. Every threshold across five notebooks is local, and the coexisting latch-and-cycle
  case from the second notebook remains unaddressed.
